# Games operations sample: extract inspection findings with AI functions (path A)

Reads the inspection report PDFs in the lakehouse with Fabric AI functions, turns every finding into a row of the
`finding` table, scores the extraction against the answer key, and writes the table. The ontology then gets a
**Finding** entity type bound to it, so findings can be counted, filtered and joined to venues, assets and sessions.

The same PDFs are indexed separately for Foundry IQ (path B, notebook 9), which answers "what exactly does the
report say" with citations. This notebook answers "which, how many and related to what".

**Before running**
- `01_load_lakehouse.ipynb` has run, and this notebook has `GamesLakehouse` attached as its default lakehouse.
- Upload the sample's `generator/` folder to `Files/games_ops/generator/`, including `schema.py`.
  Importing this notebook does not upload those Python files. `schema` is a local sample module, not a pip package.
- Upload the generated inspection PDFs to `Files/games_ops/output/documents/inspections/` and the answer key
  to `Files/games_ops/output/eval/ground_truth_findings.csv`. Adjust the parameters below if you used another layout.
- Paid Fabric capacity (F2 or higher) and the tenant setting for Copilot and other features powered by Azure
  OpenAI; depending on the region, the cross-geo processing setting too.
- Fabric Runtime 1.3 or later. AI functions use the built-in model (gpt-5-mini by default); usage counts against
  the capacity and shows in the Fabric Capacity Metrics app.
- On Runtime 2.0, uncomment the install line in the first code cell and the three temporary settings at the end
  of the imports cell. Run the install cell first: it restarts Python, so run the parameters afterwards.
  See the [Runtime 2.0 setup](https://learn.microsoft.com/fabric/data-science/ai-functions/runtime-2-0-workarounds#configure-pandas-ai-functions).

In [ ]:
# Fabric Runtime 2.0 only: uncomment the next line. Run this cell separately; it restarts Python.
# %pip install -q nest_asyncio

In [ ]:
# Parameters
DOC_DIR = "/lakehouse/default/Files/games_ops/output/documents/inspections"
GROUND_TRUTH = "/lakehouse/default/Files/games_ops/output/eval/ground_truth_findings.csv"
GEN_DIR = "/lakehouse/default/Files/games_ops/generator"
MAX_FILES = None          # e.g. 5 for a quick, cheap trial run; None = all inspection reports
WRITE_TABLE = True        # write the extracted rows to the lakehouse table `finding`

In [ ]:
import os, re, sys, json

schema_path = os.path.join(GEN_DIR, "schema.py")
if not os.path.isfile(schema_path):
    raise FileNotFoundError(
        f"Missing sample module: {schema_path}. Attach GamesLakehouse as this notebook's default lakehouse "
        "and upload the sample's generator folder to Files/games_ops/generator, or correct GEN_DIR "
        "to match your upload location. Do not pip install schema; it is not the sample module."
    )
sys.path.insert(0, GEN_DIR)
from schema import LAKEHOUSE_TABLES

import pandas as pd
import synapse.ml.aifunc as aifunc

# Fabric Runtime 2.0 only: uncomment these temporary AI functions settings.
# aifunc.default_conf.model_deployment_name = "gpt-5-mini"
# aifunc.default_conf.reasoning_effort = "low"
# aifunc.default_conf.temperature = None

## 1. List the inspection reports

In [ ]:
paths = list(aifunc.list_file_paths(DOC_DIR))
docs = pd.DataFrame({"file_path": paths})
docs["document_id"] = docs.file_path.map(lambda p: os.path.splitext(os.path.basename(str(p)))[0])
docs = docs.sort_values("document_id").reset_index(drop=True)
if MAX_FILES:
    docs = docs.head(MAX_FILES)
print(f"{len(docs)} inspection reports")
display(docs.head())

## 2. Define what to extract

One label per report-level field, and one label for the findings. The findings label is an object with a fixed
schema, so every finding comes back with the same fields; without `max_items` the function returns every
finding in the report as a list.

In [ ]:
finding_schema = {
    "properties": {
        "finding_id": {"type": "string", "description": "Finding ID as printed, for example F-0034-1"},
        "severity": {"type": "string", "enum": ["Low", "Medium", "High"]},
        "category": {"type": "string", "description": "Category column, for example Technical systems"},
        "asset_id": {"type": ["string", "null"], "description": "Asset code such as V10-AS-018; null when shown as -"},
        "description": {"type": "string", "description": "The Observation text"},
        "recommendation": {"type": "string", "description": "The Recommendation text, without the status and due date in brackets"},
        "finding_status": {"type": "string", "enum": ["Open", "Closed"]},
        "due_date": {"type": "string", "description": "Due date in YYYY-MM-DD format"},
    },
    "required": ["finding_id", "severity", "category", "asset_id", "description", "recommendation",
                 "finding_status", "due_date"],
    "additionalProperties": False,
}

labels = [
    aifunc.ExtractLabel("venue_id", description="Venue code shown in brackets after the venue name, for example V10",
                        type="string", max_items=1),
    aifunc.ExtractLabel("inspection_date", description="Inspection date in YYYY-MM-DD format",
                        type="string", max_items=1),
    aifunc.ExtractLabel("finding", description="One item per row of the Findings table in the report",
                        type="object", properties=finding_schema),
]

## 3. Extract

In [ ]:
# This code uses AI. Always review output for mistakes.
extracted = docs["file_path"].ai.extract(*labels, column_type="path")
extracted.insert(0, "document_id", docs["document_id"].values)
display(extracted.head())
try:
    display(extracted.ai.stats)
except Exception as e:
    print("Usage statistics not available:", e)

## 4. Turn the results into one row per finding

In [ ]:
def as_list(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []
    if isinstance(v, str):
        try:
            v = json.loads(v)
        except ValueError:
            return []
    if isinstance(v, dict):
        return [v]
    return [x for x in v if isinstance(x, dict) or isinstance(x, str)]

def scalar(v):
    if isinstance(v, (list, tuple)):
        v = v[0] if v else None
    return None if v is None or (isinstance(v, float) and pd.isna(v)) else str(v).strip()

def clean_id(v):
    v = scalar(v)
    return None if v in (None, "", "-", "null", "None") else v

rows = []
for _, r in extracted.iterrows():
    for f in as_list(r["finding"]):
        if isinstance(f, str):
            f = json.loads(f)
        rows.append(dict(
            finding_id=scalar(f.get("finding_id")), document_id=r["document_id"], venue_id=scalar(r["venue_id"]),
            asset_id=clean_id(f.get("asset_id")), inspection_date=scalar(r["inspection_date"]),
            severity=scalar(f.get("severity")), category=scalar(f.get("category")),
            description=scalar(f.get("description")),
            recommendation=re.sub(r"\s*\(Status:.*\)\s*$", "", scalar(f.get("recommendation")) or ""),
            finding_status=scalar(f.get("finding_status")), due_date=scalar(f.get("due_date"))))

cols = [c for c, _, _ in LAKEHOUSE_TABLES["finding"]]
findings = pd.DataFrame(rows, columns=cols)
for c in ("inspection_date", "due_date"):
    findings[c] = pd.to_datetime(findings[c], errors="coerce").dt.date
print(f"{len(findings)} findings extracted from {len(extracted)} reports")
display(findings.head(10))

## 5. Score the extraction against the answer key

The generator wrote every finding into the PDFs and also into `ground_truth_findings.csv`. A finding counts as
found when its ID matches; each field is then compared. Free-text fields count as correct when at least 80 percent
of the words match.

In [ ]:
truth = pd.read_csv(GROUND_TRUTH, dtype=str, keep_default_na=False)
truth = truth.where(truth != "", None)
truth = truth[truth.document_id.isin(docs.document_id)]
for c in ("inspection_date", "due_date"):
    truth[c] = pd.to_datetime(truth[c]).dt.date

found = findings.merge(truth, on="finding_id", how="inner", suffixes=("_x", "_t"))
recall = len(found) / len(truth) if len(truth) else 0
precision = len(found) / len(findings) if len(findings) else 0

def words(s):
    return set(re.findall(r"[a-z0-9]+", str(s or "").lower()))

def same(field, a, b):
    if field in ("description", "recommendation"):
        wa, wb = words(a), words(b)
        return len(wa & wb) / max(len(wb), 1) >= 0.8
    return (a or None) == (b or None) or str(a).lower() == str(b).lower()

fields = ["document_id", "venue_id", "asset_id", "inspection_date", "severity", "category", "description",
          "recommendation", "finding_status", "due_date"]
score = pd.DataFrame([dict(field=f, accuracy=sum(same(f, x, t) for x, t in zip(found[f + "_x"], found[f + "_t"]))
                           / max(len(found), 1)) for f in fields])
print(f"Findings in the answer key: {len(truth)}  extracted: {len(findings)}  matched: {len(found)}")
print(f"Recall {recall:.0%}  precision {precision:.0%}")
display(score.style.format({"accuracy": "{:.0%}"}))

wrong = found[[not same(f, x, t) for f, x, t in zip(["severity"] * len(found), found.severity_x, found.severity_t)]]
missed = truth[~truth.finding_id.isin(findings.finding_id)]
if len(missed):
    print("Not extracted:"); display(missed[["finding_id", "document_id", "severity", "category"]])
if len(wrong):
    print("Wrong severity:"); display(wrong[["finding_id", "severity_x", "severity_t"]])

Severity, status, asset and due date drive the `SessionAtRisk` rule, so check those first. If accuracy is low,
tighten the field descriptions in step 2 and run again. Extraction results can differ between runs.

## 6. Write the `finding` table

In [ ]:
if WRITE_TABLE:
    from pyspark.sql import types as T
    spark_types = {"string": T.StringType(), "date": T.DateType()}
    schema = T.StructType([T.StructField(c, spark_types[t], True) for c, t, _ in LAKEHOUSE_TABLES["finding"]])
    out = findings.dropna(subset=["finding_id"]).drop_duplicates("finding_id")
    out = out.astype(object).where(out.notna(), None)
    spark.createDataFrame(out[cols].values.tolist(), schema=schema) \
        .write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("finding")
    print(f"Wrote {len(out)} rows to the finding table")
    display(spark.sql("SELECT severity, finding_status, COUNT(*) AS findings FROM finding GROUP BY ALL ORDER BY ALL"))

## 7. Add Finding to the semantic model and the ontology

1. Run `03_build_semantic_model.ipynb` again (**Run all**). It adds the `finding` table, its key column and its three
   relationships to the semantic model; the rest stays as it is.
2. In the ontology, open the **Ontology agent** and send **prompt 3** from `ontology/ontology-agent-prompts.md`
   (Plan mode, check the preview, then Act mode). It adds the Finding entity type, its three relationships and the
   `SessionAtRisk` rule.
3. Refresh the ontology and run the check cell at the end of `03_build_semantic_model.ipynb`.
4. Test with evaluation question Q11: *Which scheduled sessions are at risk from open High-severity findings on
   Critical assets, and what did the inspector recommend?*

When the inspection PDFs change, run this notebook again and refresh the ontology; the table name and the
ontology bindings stay the same.